# ROC、PR、AUC 与阈值：从排序逐点手算

**面试问题：ROC-AUC 和 PR-AUC 分别回答什么，为什么阈值不能靠 0.5 拍脑袋？**

## 回答主线

1. ROC 曲线扫描阈值时画 TPR 对 FPR，衡量正负样本的整体排序能力。
2. PR 曲线画 Precision 对 Recall，在正例稀少时更直接反映告警质量。
3. AUC 与单一阈值无关，不能替代上线混淆矩阵和成本分析。
4. 计算曲线时同分样本必须作为一组跨过阈值，否则结果依赖输入顺序。
5. PR-AUC 有梯形面积和 Average Precision 等定义，报告时要注明。
6. 阈值应在验证集按容量、Recall 下限或业务成本选择，再锁定到测试集。

## 真实案例

12 笔支付交易只有 3 笔欺诈，模型给出可读风险分数，其中两笔同分。我们从零按唯一分数扫描混淆矩阵，计算 ROC 梯形 AUC、PR 梯形 AUC 和 Average Precision，并按审核容量最多 4 笔选择阈值。使用可读的离线教学数据解释机制，指标不能外推为线上收益。

### 输入预览：十二笔交易、三笔欺诈与并列分数

In [1]:
transactions = [  # 构造十二笔带分数和标签的交易。
    {"id": "T1", "amount": 1200, "score": 0.92, "label": 1},  # 高分欺诈。
    {"id": "T2", "amount": 80, "score": 0.88, "label": 0},  # 高分误报。
    {"id": "T3", "amount": 950, "score": 0.81, "label": 1},  # 第二笔欺诈。
    {"id": "T4", "amount": 60, "score": 0.74, "label": 0},  # 正常交易。
    {"id": "T5", "amount": 700, "score": 0.68, "label": 1},  # 边界欺诈。
    {"id": "T6", "amount": 55, "score": 0.68, "label": 0},  # 与 T5 同分的正常交易。
    {"id": "T7", "amount": 45, "score": 0.55, "label": 0},  # 中分正常。
    {"id": "T8", "amount": 35, "score": 0.43, "label": 0},  # 低分正常。
    {"id": "T9", "amount": 28, "score": 0.32, "label": 0},  # 低分正常。
    {"id": "T10", "amount": 20, "score": 0.21, "label": 0},  # 低分正常。
    {"id": "T11", "amount": 15, "score": 0.12, "label": 0},  # 极低分正常。
    {"id": "T12", "amount": 10, "score": 0.05, "label": 0},  # 最低分正常。
]  # 完成评估集。
print("交易  amount  score  label")  # 输出输入表头。
for row in transactions:  # 逐交易展示风险排序。
    print(f"{row['id']:<4} {row['amount']:>6}  {row['score']:.2f}    {row['label']}")  # 展示 0.68 并列组。
print(f"正例率={sum(row['label'] for row in transactions) / len(transactions):.1%}")  # 输出类别基准。

交易  amount  score  label
T1     1200  0.92    1
T2       80  0.88    0
T3      950  0.81    1
T4       60  0.74    0
T5      700  0.68    1
T6       55  0.68    0
T7       45  0.55    0
T8       35  0.43    0
T9       28  0.32    0
T10      20  0.21    0
T11      15  0.12    0
T12      10  0.05    0
正例率=25.0%


## Baseline 基线：只报 0.5 阈值 Accuracy

In [2]:
def confusion_at(threshold):  # 在指定阈值计算四格表。
    predictions = [int(row["score"] >= threshold) for row in transactions]  # 将分数转换为二值告警。
    true_positive = sum(prediction == 1 and row["label"] == 1 for prediction, row in zip(predictions, transactions))  # 统计 TP。
    false_positive = sum(prediction == 1 and row["label"] == 0 for prediction, row in zip(predictions, transactions))  # 统计 FP。
    false_negative = sum(prediction == 0 and row["label"] == 1 for prediction, row in zip(predictions, transactions))  # 统计 FN。
    true_negative = sum(prediction == 0 and row["label"] == 0 for prediction, row in zip(predictions, transactions))  # 统计 TN。
    precision = true_positive / max(1, true_positive + false_positive)  # 计算 Precision。
    recall = true_positive / max(1, true_positive + false_negative)  # 计算 Recall/TPR。
    false_positive_rate = false_positive / max(1, false_positive + true_negative)  # 计算 FPR。
    accuracy = (true_positive + true_negative) / len(transactions)  # 计算 Accuracy。
    return {"threshold": threshold, "tp": true_positive, "fp": false_positive, "fn": false_negative, "tn": true_negative, "precision": precision, "recall": recall, "fpr": false_positive_rate, "accuracy": accuracy, "alerts": true_positive + false_positive}  # 返回完整指标。

half_metrics = confusion_at(0.5)  # 评估固定 0.5 阈值。
all_negative_accuracy = sum(row["label"] == 0 for row in transactions) / len(transactions)  # 计算全负类 Accuracy。
print("0.5 阈值：", half_metrics)  # 展示单点混淆矩阵。
print(f"全预测正常 Accuracy={all_negative_accuracy:.1%}，0.5 Accuracy={half_metrics['accuracy']:.1%}")  # 说明 Accuracy 对少数类不敏感。

0.5 阈值： {'threshold': 0.5, 'tp': 3, 'fp': 4, 'fn': 0, 'tn': 5, 'precision': 0.42857142857142855, 'recall': 1.0, 'fpr': 0.4444444444444444, 'accuracy': 0.6666666666666666, 'alerts': 7}
全预测正常 Accuracy=75.0%，0.5 Accuracy=66.7%


### 核心实现：按唯一分数成组扫描 ROC 与 PR

In [3]:
unique_thresholds = [float("inf")] + sorted({row["score"] for row in transactions}, reverse=True) + [float("-inf")]  # 构造含两个端点的唯一阈值。
curve_rows = [confusion_at(threshold) for threshold in unique_thresholds]  # 一次跨过全部同分样本并计算曲线点。
print("threshold  alerts  TP FP FN TN  FPR   TPR   Precision")  # 输出曲线中间量表头。
for row in curve_rows:  # 逐唯一阈值展示混淆矩阵。
    threshold_text = f"{row['threshold']:.2f}" if abs(row["threshold"]) != float("inf") else str(row["threshold"])  # 格式化无穷端点。
    print(f"{threshold_text:>8} {row['alerts']:>7}  {row['tp']:>2} {row['fp']:>2} {row['fn']:>2} {row['tn']:>2}  {row['fpr']:.3f} {row['recall']:.3f} {row['precision']:.3f}")  # 展示 0.68 一次增加两条告警。

threshold  alerts  TP FP FN TN  FPR   TPR   Precision
     inf       0   0  0  3  9  0.000 0.000 0.000
    0.92       1   1  0  2  9  0.000 0.333 1.000
    0.88       2   1  1  2  8  0.111 0.333 0.500
    0.81       3   2  1  1  8  0.111 0.667 0.667
    0.74       4   2  2  1  7  0.222 0.667 0.500
    0.68       6   3  3  0  6  0.333 1.000 0.500
    0.55       7   3  4  0  5  0.444 1.000 0.429
    0.43       8   3  5  0  4  0.556 1.000 0.375
    0.32       9   3  6  0  3  0.667 1.000 0.333
    0.21      10   3  7  0  2  0.778 1.000 0.300
    0.12      11   3  8  0  1  0.889 1.000 0.273
    0.05      12   3  9  0  0  1.000 1.000 0.250
    -inf      12   3  9  0  0  1.000 1.000 0.250


## 结果解读：手算 ROC-AUC、PR-AUC 与 AP

In [4]:
def trapezoid_area(points):  # 对按 x 递增的点做梯形积分。
    area = 0.0  # 初始化面积。
    for left, right in zip(points[:-1], points[1:]):  # 遍历相邻线段。
        width = right[0] - left[0]  # 计算横轴增量。
        area += width * (left[1] + right[1]) / 2  # 累加梯形面积。
    return area  # 返回总面积。

roc_points = [(row["fpr"], row["recall"]) for row in curve_rows]  # 构造 FPR-TPR 点。
roc_auc = trapezoid_area(roc_points)  # 计算 ROC 梯形面积。
pr_points = [(row["recall"], row["precision"]) for row in curve_rows]  # 构造 Recall-Precision 点。
pr_auc_trapezoid = trapezoid_area(pr_points)  # 计算梯形 PR-AUC。
average_precision = 0.0  # 初始化阶梯式 AP。
for previous, current in zip(curve_rows[:-1], curve_rows[1:]):  # 遍历 Recall 增量。
    recall_gain = current["recall"] - previous["recall"]  # 计算当前阈值新增 Recall。
    average_precision += recall_gain * current["precision"]  # 用当前 Precision 加权 Recall 增量。
print(f"ROC-AUC={roc_auc:.4f}，PR-AUC(trapezoid)={pr_auc_trapezoid:.4f}，Average Precision={average_precision:.4f}")  # 报告三种面积并注明定义。
print(f"随机排序 ROC 基线=0.5，随机 PR 基线约等于正例率={sum(row['label'] for row in transactions) / len(transactions):.3f}")  # 给出可解释基准。
print("解读：ROC-AUC 看整体排序，PR 更关注正告警纯度；AP 与梯形 PR-AUC 数值不同并非计算错误，而是积分定义不同。")  # 解释指标差异。

ROC-AUC=0.8704，PR-AUC(trapezoid)=0.5278，Average Precision=0.7222
随机排序 ROC 基线=0.5，随机 PR 基线约等于正例率=0.250
解读：ROC-AUC 看整体排序，PR 更关注正告警纯度；AP 与梯形 PR-AUC 数值不同并非计算错误，而是积分定义不同。


## 失败案例：逐行跨过并列分数让 AUC 依赖输入顺序

In [5]:
def unsafe_rowwise_roc(rows):  # 错误地对同分交易逐行更新 ROC。
    positives = sum(row["label"] for row in rows)  # 统计正例总数。
    negatives = len(rows) - positives  # 统计负例总数。
    true_positive = 0  # 初始化累计 TP。
    false_positive = 0  # 初始化累计 FP。
    points = [(0.0, 0.0)]  # 从原点开始。
    for row in sorted(rows, key=lambda item: -item["score"]):  # 稳定排序会保留输入中的同分次序。
        true_positive += row["label"]  # 当前正例增加 TP。
        false_positive += 1 - row["label"]  # 当前负例增加 FP。
        points.append((false_positive / negatives, true_positive / positives))  # 每行都追加 ROC 点。
    return trapezoid_area(points)  # 返回次序敏感面积。

original_unsafe_auc = unsafe_rowwise_roc(transactions)  # 使用 T5 正例在 T6 负例之前的次序。
swapped_transactions = transactions.copy()  # 复制交易列表以交换并列样本。
swapped_transactions[4], swapped_transactions[5] = swapped_transactions[5], swapped_transactions[4]  # 仅交换 0.68 同分两行。
swapped_unsafe_auc = unsafe_rowwise_roc(swapped_transactions)  # 重新计算错误 AUC。
capacity_candidates = [row for row in curve_rows if row["alerts"] <= 4]  # 过滤人工每日最多审核四笔的阈值。
capacity_choice = max(capacity_candidates, key=lambda row: (row["recall"], row["precision"], row["threshold"]))  # 在容量内最大化 Recall。
print(f"同分原次序 unsafe AUC={original_unsafe_auc:.4f}，交换后={swapped_unsafe_auc:.4f}，正确成组AUC={roc_auc:.4f}")  # 展示次序依赖。
print("审核容量<=4 的阈值选择：", capacity_choice)  # 展示阈值来自运营约束。
print("生产边界：曲线库也需核对 sample_weight、正类定义、ties 和插值；阈值只用验证集选，测试集一次报告并监控漂移。")  # 总结工程边界。

同分原次序 unsafe AUC=0.8889，交换后=0.8519，正确成组AUC=0.8704
审核容量<=4 的阈值选择： {'threshold': 0.81, 'tp': 2, 'fp': 1, 'fn': 1, 'tn': 8, 'precision': 0.6666666666666666, 'recall': 0.6666666666666666, 'fpr': 0.1111111111111111, 'accuracy': 0.8333333333333334, 'alerts': 3}
生产边界：曲线库也需核对 sample_weight、正类定义、ties 和插值；阈值只用验证集选，测试集一次报告并监控漂移。


## 回归测试：最后只保护端点、面积、Ties 与容量

In [6]:
assert roc_points[0] == (0.0, 0.0) and roc_points[-1] == (1.0, 1.0)  # 验证 ROC 曲线包含两个端点。
assert 0.5 < roc_auc <= 1.0 and average_precision > 0.25  # 验证排序优于随机基线。
assert original_unsafe_auc != swapped_unsafe_auc  # 验证逐行处理同分的错误实现依赖输入顺序。
assert sum(row["threshold"] == 0.68 for row in curve_rows) == 1 and confusion_at(0.68)["alerts"] == 6  # 验证并列分数被一次跨过。
assert capacity_choice["alerts"] <= 4 and capacity_choice["recall"] >= 2 / 3  # 验证容量阈值满足审核上限并召回至少两笔欺诈。
print("回归测试通过：ROC端点、面积基准、Ties反例、成组阈值和审核容量均成立。")  # 用少量断言总结指标合同。

回归测试通过：ROC端点、面积基准、Ties反例、成组阈值和审核容量均成立。
